In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os
from datetime import datetime, timedelta
import shutil

from src.gtfs.helper import expand_timetable

# Network

In [2]:
# Load Network Files
network_name = "11-500"
network_path = f"data/network/{network_name}/"
node_df = pd.read_csv(network_path + "nodes.csv")
link_df = pd.read_csv(network_path + "edges.csv")

node_id_list = node_df['node_index'].tolist()
print(f"Number of nodes: {len(node_id_list)}")

left_node_id_list = [i for i in range(0, len(node_id_list)//2)]
right_node_id_list = [i for i in range(len(node_id_list)//2, len(node_id_list))]
print(f"Number of left nodes: {len(left_node_id_list)}")
print(f"Number of right nodes: {len(right_node_id_list)}")

left_mobility_node_id_list = [116, 117, 118, 119, 120]
right_mobility_node_id_list = [237, 238, 239, 240, 241]

Number of nodes: 242
Number of left nodes: 121
Number of right nodes: 121


In [3]:
# Load Distance Matrix
distance_matrix = np.load(network_path + "dist_matrix.npy")
print(f"Distance matrix shape: {distance_matrix.shape}")

Distance matrix shape: (242, 242)


In [4]:
# Load Travel Time Matrix
tt_matrix = np.load(network_path + "tt_matrix.npy")
print(f"Travel time matrix shape: {tt_matrix.shape}")

Travel time matrix shape: (242, 242)


# Create General Files

In [5]:
general_output_path = "data/gtfs/general/"
if not os.path.exists(general_output_path):
    os.makedirs(general_output_path)

# Create stations_fp.csv
stations_fp_df = pd.DataFrame(columns=['station_id','station_name','station_lat','station_lon','stops_included','station_stop_transfer_times','num_stops_included'])
for index, node in node_df.iterrows():
    station_id = node['node_index']
    station_name = f'B-{station_id}'
    station_lat = node['pos_y']
    station_lon = node['pos_x']

    stops_included = "['{}-0';'{}-1';'{}-2';'{}-3';'{}-4';'{}-5']".format(station_id, station_id, station_id, station_id, station_id, station_id)
    station_stop_transfer_times = '[0;0;0;0;0;0]'
    num_stops_included = 6

    new_row = pd.DataFrame({
        'station_id': [station_id],
        'station_name': [station_name],
        'station_lat': [station_lat],
        'station_lon': [station_lon],
        'stops_included': [stops_included],
        'station_stop_transfer_times': [station_stop_transfer_times],
        'num_stops_included': [num_stops_included]
    })

    stations_fp_df = pd.concat([stations_fp_df, new_row], ignore_index=True)

# Remove non-stop nodes
left_non_stops_node_id_list = [1,2,3,4,6,7,8,13,14,15,17,18,21,22,31,32,33,34,37,42,43,44,45,50,52,53,62,63,65,70,71,72,73,78,81,82,83,84,93,94,97,98,100,101,102,107,108,109,111,112,113,114]
right_non_stops_node_id_list = [i + 121 for i in left_non_stops_node_id_list]
non_stop_node_id_list = left_non_stops_node_id_list + right_non_stops_node_id_list

stations_fp_df = stations_fp_df[~stations_fp_df['station_id'].isin(non_stop_node_id_list)].reset_index(drop=True)

# Change 120 to A, 241 to B
stations_fp_df.loc[stations_fp_df['station_id'] == 120, 'station_id'] = 'MH-L'
stations_fp_df.loc[stations_fp_df['station_id'] == 'MH-L', 'station_name'] = 'Mobility Hub Left'
stations_fp_df.loc[stations_fp_df['station_id'] == 'MH-L', 'stops_included'] = "['MH-L-0';'MH-L-1';'MH-L-2';'MH-L-3';'MH-L-4';'MH-L-5']"

stations_fp_df.loc[stations_fp_df['station_id'] == 241, 'station_id'] = 'MH-R'
stations_fp_df.loc[stations_fp_df['station_id'] == 'MH-R', 'station_name'] = 'Mobility Hub Right'
stations_fp_df.loc[stations_fp_df['station_id'] == 'MH-R', 'stops_included'] = "['MH-R-0';'MH-R-1';'MH-R-2';'MH-R-3';'MH-R-4';'MH-R-5']"

# Save stations_fp.txt
stations_fp_df.to_csv(general_output_path + "stations_fp.txt", index=False)

In [6]:
# Create stops_fp.txt
stops_fp_df = pd.DataFrame(columns=['stop_id'])
for index, row in stations_fp_df.iterrows():
    # Convert stops_included string to list
    s = row['stops_included']
    if isinstance(s, str):
        s_clean = s.strip()
        if s_clean.startswith('[') and s_clean.endswith(']'):
            s_clean = s_clean[1:-1]
        s_clean = s_clean.replace("'", "").replace('"', '').strip()
        if ';' in s_clean:
            parts = [p.strip() for p in s_clean.split(';') if p.strip()]
        elif ',' in s_clean:
            parts = [p.strip() for p in s_clean.split(',') if p.strip()]
        elif s_clean == '':
            parts = []
        else:
            parts = [s_clean]
    else:
        parts = list(s) if hasattr(s, '__iter__') and not isinstance(s, str) else [s]
    row['stops_included'] = ';'.join(parts)
    stop_ids = row['stops_included']
    for stop_id in stop_ids.split(';'):
        stops_fp_df = pd.concat([stops_fp_df, pd.DataFrame({'stop_id': [stop_id]})], ignore_index=True)
stops_fp_df.to_csv(os.path.join(general_output_path, "stops_fp.txt"), index=False)

In [7]:
# Prepare agency_fp.txt
agency_df = pd.DataFrame({
    'agency_id': [0, 1, 2, 3],
    'agency_name': ['train', 'bus-basic', 'bus-ring', 'bus-diagonal'],
})
agency_df.to_csv(os.path.join(general_output_path, "agency_fp.txt"), index=False)

In [8]:
# Create calendar_fp.txt
calendar_df = pd.DataFrame({
    'service_id': [0],
    'start_date': [20000101],
    'end_date': [20991231],
    'monday': [1],
    'tuesday': [1],
    'wednesday': [1],
    'thursday': [1],
    'friday': [1],
    'saturday': [1],
    'sunday': [1],
})
calendar_df.to_csv(os.path.join(general_output_path, "calendar_fp.txt"), index=False)

In [9]:
# Create street_station_transfers_fp.txt
all_station_ids = stations_fp_df['station_id'].tolist()

street_station_transfers_fp_df = pd.DataFrame({
    'node_id': all_station_ids,
    'closest_station_id': all_station_ids,
    'street_station_transfer_time': 60
})

# Change node id from A and B back to 120 and 241
street_station_transfers_fp_df.loc[street_station_transfers_fp_df['node_id'] == 'MH-L', 'node_id'] = 120
street_station_transfers_fp_df.loc[street_station_transfers_fp_df['node_id'] == 'MH-R', 'node_id'] = 241

street_station_transfers_fp_df.to_csv(os.path.join(general_output_path, "street_station_transfers_fp.txt"), index=False)

In [10]:
# Create transfers_fp.txt
# For each station, the transfer time between its stops is 30 seconds
transfers_fp_df = pd.DataFrame(columns=['from_stop_id', 'to_stop_id', 'min_transfer_time'])
for index, row in stations_fp_df.iterrows():
    stops_included = row['stops_included'].split(';')
    for i in range(len(stops_included)):
        for j in range(len(stops_included)):
            if i != j:
                new_row = pd.DataFrame({
                    'from_stop_id': [stops_included[i]],
                    'to_stop_id': [stops_included[j]],
                    'min_transfer_time': [30]
                })
                transfers_fp_df = pd.concat([transfers_fp_df, new_row], ignore_index=True)
transfers_fp_df.to_csv(os.path.join(general_output_path, "transfers_fp.txt"), index=False)

In [11]:
# # Create routes_fp.txt
# routes_df = pd.DataFrame({
#     'route_id': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
#     'route_short_name': [
#         'train', 
#         'bus-basic-we-left', 'bus-basic-ns-left', 'bus-basic-we-right', 'bus-basic-ns-right', 
#         'bus-ring-left', 'bus-ring-right',
#         'bus-diagonal-sw-left', 'bus-diagonal-se-left', 'bus-diagonal-sw-right', 'bus-diagonal-se-right'],
#     'route_desc': ['train', 'bus', 'bus', 'bus', 'bus', 'bus', 'bus', 'bus', 'bus', 'bus', 'bus'],
# })
# routes_df.to_csv(os.path.join(general_output_path, "routes_fp.txt"), index=False)

# Create Train GTFS

In [12]:
# Train headway (min)
train_headway_list = [10, 20, 30]

# Train turnaround time (min)
train_turnaround_time = 5

# Train trip time (min) between left and right mobility hubs
train_trip_time = 25

# Prepare output path
train_output_path = f"data/gtfs/train"
if not os.path.exists(train_output_path):
    os.makedirs(train_output_path)

STUDY_START = "00:00:01"
STUDY_END = "08:00:00"

In [13]:
# Create stop_times_fp.txt
stop_times_fp_df = pd.DataFrame(columns=['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence'])

trip_times_0 = {
    'trip_id': '0-0', # route_id-direction
    'stop_id': ['MH-L-0', 'MH-R-0'],
    'arrival_time': ['00:55:00', '01:25:00'],
    'departure_time': ['01:00:00', '01:30:00'],
    'stop_sequence': [1, 2]
}

trip_times_1 = {
    'trip_id': '0-1', # route_id-direction
    'stop_id': ['MH-R-0', 'MH-L-0'],
    'arrival_time': ['00:55:00', '01:25:00'],
    'departure_time': ['01:00:00', '01:30:00'],
    'stop_sequence': [1, 2]
}

base_schedules = [
    trip_times_0,
    trip_times_1,
]

for headway in train_headway_list:
    all_schedules_list = []
    for schedule_data in base_schedules:
        expanded_schedule = expand_timetable(schedule_data, STUDY_START, STUDY_END, headway)
        all_schedules_list.append(expanded_schedule)

    final_timetable = pd.concat(all_schedules_list, ignore_index=True)
    final_timetable['arrival_time'] = final_timetable['arrival_time'].dt.strftime('%H:%M:%S')
    final_timetable['departure_time'] = final_timetable['departure_time'].dt.strftime('%H:%M:%S')
    # Save stop_times_fp.txt
    save_dirpath = os.path.join(train_output_path, f"train_headway_{headway}", 'matched')
    os.makedirs(save_dirpath, exist_ok=True)
    final_timetable.to_csv(os.path.join(save_dirpath, "stop_times_fp.txt"), index=False)

    # Create trips_fp.txt
    trips_fp_df = pd.DataFrame(columns=['trip_id','route_id','service_id','direction_id'])

    all_trip_ids = final_timetable['trip_id'].unique().tolist()
    for trip_id in all_trip_ids:
        route_id = int(trip_id.split('-')[0])
        direction_id = int(trip_id.split('-')[1])
        new_row = pd.DataFrame({
            'trip_id': [trip_id],
            'route_id': [route_id],
            'service_id': [0],
            'direction_id': [direction_id]
        })
        trips_fp_df = pd.concat([trips_fp_df, new_row], ignore_index=True)
    trips_fp_df.to_csv(os.path.join(save_dirpath, "trips_fp.txt"), index=False)

    # Create routes_fp.txt
    routes_df = pd.DataFrame({
        'route_id': [0],
        'route_short_name': ['train'],
        'route_desc': ['train'],
    })
    routes_df.to_csv(os.path.join(save_dirpath, "routes_fp.txt"), index=False)

    # Copy all general files to the specific train headway folder
    general_files = [
        "agency_fp.txt",
        "calendar_fp.txt",
        "stations_fp.txt",
        "stops_fp.txt",
        "street_station_transfers_fp.txt",
        "transfers_fp.txt"
    ]
    for file_name in general_files:
        src_path = os.path.join(general_output_path, file_name)
        dst_path = os.path.join(save_dirpath, file_name)
        if os.path.exists(src_path):
            shutil.copy(src_path, dst_path)
            